# Qasper Long-Context Survey Standalone

This standalone notebook checks whether Qasper has enough long-context documents. It does not import from the repo.


In [ ]:
# Simple Kaggle/Colab setup. Run this cell first.
# Do not force reinstall Kaggle's scientific stack; only install packages if missing.
import importlib.metadata as importlib_metadata
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "datasets": "datasets",
    "pyarrow": "pyarrow",
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers",
    "torch": "torch",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "tqdm": "tqdm",
}

missing = [package for module, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
import sklearn
import torch
from sentence_transformers import SentenceTransformer

def version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"

print("Dependency check OK:")
print("python", sys.version.split()[0])
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)
print("transformers", version("transformers"))
print("sentence-transformers", version("sentence-transformers"))


In [ ]:
from __future__ import annotations

import json
import math
import re
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import mean, median
from typing import Any, Iterable

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
QASPER_REVISION = "cc58ffb39db7ff6ce1951e28e029996bf499304e"
QASPER_BASE_URL = f"https://huggingface.co/datasets/allenai/qasper/resolve/{QASPER_REVISION}/qasper"
QASPER_PARQUET_FILES = {
    "train": f"{QASPER_BASE_URL}/qasper-train.parquet",
    "validation": f"{QASPER_BASE_URL}/qasper-validation.parquet",
    "test": f"{QASPER_BASE_URL}/qasper-test.parquet",
}


def load_qasper(split: str = "validation"):
    if split not in QASPER_PARQUET_FILES:
        raise ValueError(f"Unknown split: {split}")
    return load_dataset("parquet", data_files={split: QASPER_PARQUET_FILES[split]}, split=split)


@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    section: str
    text: str


@dataclass(frozen=True)
class QAExample:
    doc_id: str
    question_id: str
    title: str
    question: str
    gold_answers: list[str]
    evidence: list[str]


def document_text(record: dict[str, Any]) -> str:
    parts = []
    abstract = str(record.get("abstract", "")).strip()
    if abstract:
        parts.append(abstract)

    full_text = record.get("full_text", {})
    sections = full_text.get("section_name", [])
    paragraphs_by_section = full_text.get("paragraphs", [])
    for section, paragraphs in zip(sections, paragraphs_by_section):
        section_parts = [str(section).strip()] if str(section).strip() else []
        section_parts.extend(str(paragraph).strip() for paragraph in paragraphs if str(paragraph).strip())
        if section_parts:
            parts.append("\n".join(section_parts))
    return "\n\n".join(parts)


def document_word_count(record: dict[str, Any]) -> int:
    return len(document_text(record).split())


def iter_answer_records(answers: Any) -> list[dict[str, Any]]:
    if isinstance(answers, list):
        return [answer for answer in answers if isinstance(answer, dict)]
    if not isinstance(answers, dict):
        return []

    answer_values = answers.get("answer", [])
    annotation_ids = answers.get("annotation_id", [])
    worker_ids = answers.get("worker_id", [])

    if isinstance(answer_values, dict):
        answer_values = [answer_values]
    if not isinstance(answer_values, list):
        return []

    records = []
    for index, answer_value in enumerate(answer_values):
        record = {"answer": answer_value}
        if isinstance(annotation_ids, list) and index < len(annotation_ids):
            record["annotation_id"] = annotation_ids[index]
        if isinstance(worker_ids, list) and index < len(worker_ids):
            record["worker_id"] = worker_ids[index]
        records.append(record)
    return records


def normalise_answer(answer: dict[str, Any]) -> str | None:
    data = answer.get("answer", answer)
    if data.get("unanswerable"):
        return "Unanswerable"
    if data.get("free_form_answer"):
        return str(data["free_form_answer"]).strip()
    if data.get("extractive_spans"):
        spans = [str(span).strip() for span in data["extractive_spans"] if str(span).strip()]
        if spans:
            return " ; ".join(spans)
    yes_no = data.get("yes_no")
    if yes_no is not None:
        return str(yes_no)
    return None


def normalise_evidence(answer: dict[str, Any]) -> list[str]:
    data = answer.get("answer", answer)
    evidence = data.get("evidence", answer.get("evidence", []))
    if not evidence:
        return []
    return [str(item).strip() for item in evidence if str(item).strip()]


def extract_qa_examples(record: dict[str, Any]) -> list[QAExample]:
    qas = record.get("qas", {})
    questions = qas.get("question", [])
    question_ids = qas.get("question_id", [])
    answers_list = qas.get("answers", [])

    examples: list[QAExample] = []
    for question, question_id, answers in zip(questions, question_ids, answers_list):
        gold_answers = []
        evidence = []
        for answer in iter_answer_records(answers):
            normalised = normalise_answer(answer)
            if normalised:
                gold_answers.append(normalised)
            evidence.extend(normalise_evidence(answer))
        examples.append(
            QAExample(
                doc_id=record["id"],
                question_id=question_id,
                title=record.get("title", ""),
                question=question,
                gold_answers=gold_answers,
                evidence=evidence,
            )
        )
    return examples


def chunk_words(text: str, *, chunk_size: int = 180, overlap: int = 40) -> list[str]:
    words = text.split()
    if not words:
        return []
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    chunks: list[str] = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        window = words[start : start + chunk_size]
        if window:
            chunks.append(" ".join(window))
        if start + chunk_size >= len(words):
            break
    return chunks


def build_document_chunks(record: dict[str, Any], *, chunk_size: int = 180, overlap: int = 40) -> list[Chunk]:
    full_text = record.get("full_text", {})
    sections = full_text.get("section_name", [])
    paragraphs_by_section = full_text.get("paragraphs", [])
    chunks: list[Chunk] = []
    chunk_index = 0

    abstract = record.get("abstract", "")
    for text in chunk_words(abstract, chunk_size=chunk_size, overlap=overlap):
        chunks.append(Chunk(f"{record['id']}::abstract::{chunk_index}", record["id"], record.get("title", ""), "abstract", text))
        chunk_index += 1

    for section, paragraphs in zip(sections, paragraphs_by_section):
        section_text = " ".join(str(paragraph) for paragraph in paragraphs if str(paragraph).strip())
        for text in chunk_words(section_text, chunk_size=chunk_size, overlap=overlap):
            chunks.append(Chunk(f"{record['id']}::{chunk_index}", record["id"], record.get("title", ""), str(section), text))
            chunk_index += 1
    return chunks

In [ ]:
SPLIT = "validation"
OUTPUT_DIR = "outputs/independent"
THRESHOLDS = [1000, 3000, 5000, 8000, 12000]

In [ ]:
def percentile(values: list[int], ratio: float) -> int:
    if not values:
        return 0
    return values[round((len(values) - 1) * ratio)]


def threshold_flags(word_count: int) -> dict[str, bool]:
    return {str(threshold): word_count >= threshold for threshold in THRESHOLDS}


def run_long_context_survey(split: str = SPLIT):
    dataset = load_qasper(split)
    lengths = []
    qa_counts = []
    document_rows = []
    docs_by_threshold = {threshold: 0 for threshold in THRESHOLDS}
    qa_by_threshold = {threshold: 0 for threshold in THRESHOLDS}

    for record in dataset:
        word_count = document_word_count(record)
        qa_examples = extract_qa_examples(record)
        qa_count = len(qa_examples)
        lengths.append(word_count)
        qa_counts.append(qa_count)
        document_rows.append(
            {
                "split": split,
                "doc_id": record.get("id", ""),
                "title": record.get("title", ""),
                "word_count": word_count,
                "qa_examples": qa_count,
                "thresholds": threshold_flags(word_count),
            }
        )
        for threshold in THRESHOLDS:
            if word_count >= threshold:
                docs_by_threshold[threshold] += 1
                qa_by_threshold[threshold] += qa_count

    lengths = sorted(lengths)
    total_docs = len(lengths)
    total_qa = sum(qa_counts)
    threshold_rows = [
        {
            "split": split,
            "threshold": threshold,
            "documents": docs_by_threshold[threshold],
            "document_rate": docs_by_threshold[threshold] / total_docs if total_docs else 0.0,
            "qa_examples": qa_by_threshold[threshold],
            "qa_example_rate": qa_by_threshold[threshold] / total_qa if total_qa else 0.0,
        }
        for threshold in THRESHOLDS
    ]
    summary = {
        "split": split,
        "documents": total_docs,
        "qa_examples": total_qa,
        "word_count": {
            "min": min(lengths),
            "p25": percentile(lengths, 0.25),
            "median": int(median(lengths)),
            "mean": mean(lengths),
            "p75": percentile(lengths, 0.75),
            "p90": percentile(lengths, 0.90),
            "p95": percentile(lengths, 0.95),
            "max": max(lengths),
        },
        "thresholds": {
            str(row["threshold"]): {
                key: row[key]
                for key in ("documents", "document_rate", "qa_examples", "qa_example_rate")
            }
            for row in threshold_rows
        },
    }
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = output_dir / f"qasper_long_context_survey_{split}.json"
    documents_path = output_dir / f"qasper_long_context_survey_{split}_documents.jsonl"
    thresholds_path = output_dir / f"qasper_long_context_survey_{split}_thresholds.jsonl"

    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    with documents_path.open("w", encoding="utf-8") as file:
        for row in document_rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    with thresholds_path.open("w", encoding="utf-8") as file:
        for row in threshold_rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

    summary["summary_path"] = str(summary_path)
    summary["documents_jsonl_path"] = str(documents_path)
    summary["thresholds_jsonl_path"] = str(thresholds_path)
    return summary


In [ ]:
summary = run_long_context_survey(SPLIT)
summary
